# Building CodAdapt

## Goal

Build CodAdapt from scratch to understand how the algorithm works step by step.

CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

This project rebuilds the core ideas from the simplest possible version and gradually introduces bucketization, coded memory, collisions, and adaptive codes.

---

## Version 3

Introduce adaptive codes.

Instead of assigning fixed codes, the model learns how buckets should share a smaller memory.

The model will:

- divide `Age` into buckets
- learn a code for each bucket
- allow multiple buckets to share the same memory cell
- use the learned codes to make predictions

In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

## Data preparation

We use `Age` as the only input feature and `Survived` as the target.

Missing ages are filled using the median learned from the training data.

In [3]:
target = "Survived"

age_median = train["Age"].median()

x = train["Age"].fillna(age_median).to_numpy()
y = train[target].to_numpy()

## Building the model

As before, `Age` is divided into four buckets.

This time, the buckets will share a smaller memory with only two cells.

In [4]:
cuts = np.quantile(x, [0.25, 0.50, 0.75])

bucket = np.digitize(x, cuts)

In [5]:
bucket_mean = np.array([
    y[bucket == b].mean()
    for b in range(4)
])

Buckets are assigned to one of two codes based on whether their target mean is above or below the global target mean.

This creates collisions: different buckets can share the same memory address.

In [6]:
codes = (bucket_mean >= y.mean()).astype(int)

address = codes[bucket]

In [7]:
memory = np.zeros(2)

for a in range(2):
    memory[a] = y[address == a].mean()

In [8]:
codes, memory

(array([1, 1, 0, 1]), array([0.3354232 , 0.41083916]))

A prediction is obtained by using the code as an address into memory.

In [9]:
preds = memory[address]

In [10]:
preds[:10]

array([0.41083916, 0.41083916, 0.41083916, 0.41083916, 0.41083916,
       0.3354232 , 0.41083916, 0.41083916, 0.41083916, 0.41083916])

## Prediction and submission

The test ages are converted into buckets using the same cut points learned from the training data.

Each bucket is mapped to a code, which selects the prediction from memory.

In [11]:
x_test = test["Age"].fillna(age_median).to_numpy()

bucket_test = np.digitize(x_test, cuts)
address_test = codes[bucket_test]

probabilities = memory[address_test]
preds = (probabilities >= 0.5).astype(int)

In [12]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": preds
})

submission.to_csv("submission_v3.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Notes

- This third version introduces adaptive codes and collisions.
- `Age` is still the only input feature.
- Continuous values are discretized into four buckets.
- The memory is reduced from four cells to two.
- Codes are learned from the target behavior of each bucket.
- Multiple buckets can share the same memory address.
- The learned codes are reused for the test data.
- This uses a very simple adaptive rule to explain the concept, not the full CodAdapt optimization.
- The model does not yet use multiple features, feature interactions, or shared multi-resolution encoding.
- This notebook is a simplified educational reconstruction of CodAdapt, not the full library implementation.
- CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

Future versions will introduce new components and gradually evolve the architecture.